In [5]:
import pandas as pd
import numpy as np
import os
import warnings

from scipy.signal import savgol_filter

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Load data
# -----------------------------
def load_data():

    print("Loading data...")

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


# -----------------------------
# Prepare features
# -----------------------------
def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    print("Number of spectral features:", X.shape[1])

    return X, y, X_test


# -----------------------------
# Savitzky-Golay derivative
# -----------------------------
def apply_savgol_derivative(X, X_test):

    print("\nApplying Savitzky-Golay first derivative...")

    X_deriv = savgol_filter(
        X,
        window_length=11,
        polyorder=2,
        deriv=1
    )

    X_test_deriv = savgol_filter(
        X_test,
        window_length=11,
        polyorder=2,
        deriv=1
    )

    return X_deriv, X_test_deriv


# -----------------------------
# Scale features
# -----------------------------
def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    print("Feature scaling complete")

    return X_scaled, X_test_scaled


# -----------------------------
# Cross-validation
# -----------------------------
def cross_validate(X, y):

    print("\nRunning 5-fold cross-validation...")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    rmse_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        print(f"Training fold {fold+1}")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = ElasticNet(
            alpha=1.0,
            l1_ratio=0.7,
            max_iter=100000,
            tol=1e-3,
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmse_scores.append(rmse)

        print("Fold RMSE:", rmse)

    print("\nMean CV RMSE:", np.mean(rmse_scores))


# -----------------------------
# Train final model
# -----------------------------
def train_final_model(X, y, X_test):

    print("\nTraining final model on full dataset...")

    model = ElasticNet(
        alpha=1.0,
        l1_ratio=0.7,
        max_iter=100000,
        tol=1e-3,
        random_state=42
    )

    model.fit(X, y)

    preds = model.predict(X_test)

    print("Sample predictions:", preds[:10])

    return preds


# -----------------------------
# Save submission
# -----------------------------
def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    experiment_name = "exp10_elasticnet_savgol_derivative_20260323"

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("\nSubmission saved to:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


# -----------------------------
# Main pipeline
# -----------------------------
def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = apply_savgol_derivative(X, X_test)

    X, X_test = scale_features(X, X_test)

    cross_validate(X, y)

    preds = train_final_model(X, y, X_test)

    save_submission(test, preds)


# Run experiment
main()

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Number of spectral features: 1555

Applying Savitzky-Golay first derivative...
Feature scaling complete

Running 5-fold cross-validation...
Training fold 1
Fold RMSE: 17.83557300997692
Training fold 2
Fold RMSE: 19.292570844590813
Training fold 3
Fold RMSE: 18.67496264525238
Training fold 4
Fold RMSE: 17.277309878885728
Training fold 5
Fold RMSE: 17.59074159677168

Mean CV RMSE: 18.134231595095503

Training final model on full dataset...
Sample predictions: [203.54620999 166.21511326 155.52318903 146.96093994 141.39895305
 134.98378829 125.30656503 127.18494403 123.35666303 119.12847951]

Submission saved to: ../submissions/exp10_elasticnet_savgol_derivative_20260323.csv
    0           1
0  95  203.546210
1  96  166.215113
2  97  155.523189
3  98  146.960940
4  99  141.398953
